In [16]:
from openfermion import QubitOperator, get_fermion_operator

from ofex.clifford import diagonalizing_clifford, clifford_simulation, tableau_to_pauli, str_tableau, pauli_to_tableau
from ofex.linalg.sparse_tools import expectation
from ofex.measurement import sorted_insertion
from ofex.state.chem_ref_state import hf_ground
from ofex.transforms import fermion_to_qubit_operator
from ofex.utils.chem import molecule_example, run_driver
from test_scripts.random_object import random_state_nparray



## 1. Prepare Hamiltonian and Reference State

In [17]:
# Obtain the Fermionic Hamiltonian

mol_name = "H4"

mol = molecule_example(mol_name)
mol = run_driver(mol)
fham = mol.get_molecular_hamiltonian()
fham = get_fermion_operator(fham)

In [18]:
# Fermion-to-Qubit mapping

transform = "symmetry_conserving_bravyi_kitaev"
f2q_kwargs = {"active_fermions": mol.n_electrons,
              "active_orbitals": mol.n_qubits}

n_qubits = mol.n_qubits - 2  # Two-qubit reduction

ref = hf_ground(mol, fermion_to_qubit_map=transform, **f2q_kwargs)
pham = fermion_to_qubit_operator(fham, transform, **f2q_kwargs)
p_const = pham.constant
# Make qubit hamiltonian traceless
pham = pham - p_const

## 2. Sorted Insertion

- Pauli 1-norm: Expected measurement cost when each Pauli operators are measured individually.
- Sorted Insertion: Pauli grouping technique. If the grouped operators needs to be unitary, set `anticommute=True`. Otherwise, the grouped pauli operators commute to each other and become Clifford-diagonalizable.

In [19]:
pauli_one_norm = pham.induced_norm(order=1)

si_comm = sorted_insertion(pham, anticommute=False)
si_anti = sorted_insertion(pham, anticommute=True)

si_comm_cost = sum([frag.induced_norm(order=2) for frag in si_comm])
si_anti_cost = sum([frag.induced_norm(order=2) for frag in si_anti])

print("Pauli One Norm:", pauli_one_norm)
print("Sorted Insertion Cost (commutative):", si_comm_cost)
print("Sorted Insertion Cost (anticommutative):", si_anti_cost)

Pauli One Norm: 8.546924803200866
Sorted Insertion Cost (commutative): 2.0310151017252998
Sorted Insertion Cost (anticommutative): 6.007013662956864


For commutative grouping, use `diagonalizing_clifford` for the simultaneous diagonalization of grouped Pauli operators. The example code demonstrate the fragmented measurement of
$$
\braket{\psi|\hat{H}|\psi}=\sum_j \braket{\psi|\hat{H}_j|\psi} = \sum_j \braket{\psi|\hat{C}_j^{\dagger}\hat{Z}_j\hat{C}|\psi},
$$
where $\hat{H}_j$ is a Hamiltonian fragment, $\hat{C}_j$ is corresponding diagonalizing clifford.
Here, $\ket{\psi}$ is randomly generated.

In [20]:
psi = random_state_nparray(n_qubits)
true_expectation = expectation(pham, psi, sparse=False)
frag_expectation = 0
for j, frag in enumerate(si_comm):
    tab_z_pauli, coeff, clif_hist = diagonalizing_clifford(frag, n_qubits)
    z_op = QubitOperator.accumulate(
        tableau_to_pauli(tab_z_pauli, coeff)
    )
    clif_psi = clifford_simulation(psi, clif_hist)
    expectation_j = expectation(z_op, clif_psi, sparse=False)

    if j == 1:
        tab_frag, frag_coeff = pauli_to_tableau(frag, n_qubits)
        print("The first fragment:")
        print("Fragment (frag):\n\t" + '\n\t'.join(str(frag).split('\n')))
        print("Tabulated Frag :\n\t" + '\n\t'.join(str_tableau(tab_frag).split('\n')))
        print("Clifford History (clif_hist):", clif_hist)
        print("Tabulated Z Pauli (tab_z_pauli)\n\t" + '\n\t'.join(str_tableau(tab_z_pauli).split('\n')))
        print("Z Operator (z_op):\n\t", '\n\t'.join(str(z_op).split('\n')))

    frag_expectation += expectation_j

# Added print statement for comparison
print("True Expectation:", true_expectation)
print("Fragmented Expectation:", frag_expectation)
print("Difference:", true_expectation - frag_expectation)

The first fragment:
Fragment (frag):
	-0.0254544656551172 [X0 Z1 X2] +
	-0.03872306490340444 [X0 Z1 X3] +
	0.03872306490340444 [X0 Z1 X3 Z4] +
	0.03593469149844004 [X0 Z1 Z4 X5] +
	-0.03593469149844004 [X0 Z1 X5] +
	0.05300619296799151 [X0 X2] +
	0.03872306490340444 [X0 X3] +
	-0.03872306490340444 [X0 X3 Z4] +
	-0.03593469149844004 [X0 Z4 X5] +
	0.03593469149844004 [X0 X5] +
	-0.03593469149844004 [Z1 X2 X3] +
	0.03593469149844004 [Z1 X2 X3 Z4] +
	0.03927412117061923 [Z1 X2 Z4 X5] +
	-0.03927412117061923 [Z1 X2 X5] +
	0.03593469149844004 [X2 X3] +
	-0.03593469149844004 [X2 X3 Z4] +
	-0.03927412117061923 [X2 Z4 X5] +
	0.03927412117061923 [X2 X5] +
	-0.0254544656551172 [X3 Z4 X5] +
	0.05300619296799151 [X3 X5]
Tabulated Frag)
	1 0 0 0 0 0 1 1 1 1 1 1 1 1 0 0 0 0 1 0
	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
	1 0 1 1 1 1 0 0 0 0 0 0 0 0 1 1 1 1 1 0
	0 1 0 0 0 0 1 1 1 1 0 0 0 0 1 1 1 1 0 1
	0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
	0 1 1 1 1 1 0 0 0 0 1 1 1 1 0 0 0 0 0 1
	0 0 0 0 0 0 0 0 0 0 

## 3. Theoretical Measurement Cost